# Preprocessing and feature selection

This notebook turns the findings of the EDA into a preparation the models can use, and then
chooses which columns the model sees. Every decision is measured on the
same 5 time-based splits. A change only counts when its gain is larger than 2 × SEM, the
standard error of the paired difference; a smaller gain is noise, and when two options tie
we keep the simpler one.

The choices are saved in `preprocessing.py`, which this notebook and the modelling notebook
both import.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler
from category_encoders import CountEncoder, TargetEncoder   # see the README

# constants shared with the modelling notebook
from preprocessing import DATE, GROUP, METRIC, RANDOM_STATE, TARGET

# without a scaler the logistic regression does not converge in 1000 iterations (section 4);
# the cause is known and measured, so the warning is silenced
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Starting point

The 65 rows without a date cannot be placed in time, so they are left out and the rest is
sorted by date. Every test purchase is later than every training purchase, and 94.5% of the
test rows belong to customers we already saw in training. Each validation split therefore
trains on the past and checks the purchases that come next.

The baseline fills gaps with the median and the mode, one-hot encodes the categorical
columns and uses a `StandardScaler`, with no new features. Always predicting 1 already
scores F1 0.72, because 56% of purchases are followed by another one, so that is the floor
any model has to beat.

In [2]:
# train.csv sorted by date, without the rows that have no date
from preprocessing import load_train

raw, frame = load_train("../data/train.csv")

print("rows left out for having no date:", len(raw) - len(frame))
print(f"rows in the timeline: {len(frame):,}, from {frame[DATE].iloc[0]} to {frame[DATE].iloc[-1]}")

rows left out for having no date: 65
rows in the timeline: 6,469, from 2015-07-11 to 2025-02-15


In [3]:
# only the structure of test.csv (dates and customers); none of its values is used to fit anything
test_csv = pd.read_csv("../data/test.csv", index_col="ID")
test_rows_seen = test_csv[GROUP].isin(raw[GROUP])

pd.Series({
    "training dates": f"{frame[DATE].iloc[0]} to {frame[DATE].iloc[-1]}",
    "test dates": f"{test_csv[DATE].min()} to {test_csv[DATE].max()}",
    "test customers already seen in training":
        f"{pd.Series(test_csv[GROUP].unique()).isin(raw[GROUP]).sum()} of {test_csv[GROUP].nunique()}",
    "test rows with a customer already seen":
        f"{test_rows_seen.sum()} of {len(test_csv)} ({100 * test_rows_seen.mean():.1f}%)",
}).to_frame("what the test set asks")

,what the test set asks
training dates,2015-07-11 to 2025-02-15
test dates,2025-05-24 to 2026-10-06
test customers already seen in training,184 of 206
test rows with a customer already seen,932 of 986 (94.5%)


In [4]:
# each split trains on the past and validates on what comes next
from preprocessing import time_splits

splits = time_splits(frame)

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"split {fold}: train up to {frame[DATE].iloc[train_idx].max()}"
          f" ({len(train_idx):,} rows), validate from {frame[DATE].iloc[val_idx].min()}"
          f" to {frame[DATE].iloc[val_idx].max()} ({len(val_idx):,} rows)")

split 1: train up to 2017-08-12 (1,079 rows), validate from 2017-08-12 to 2019-01-12 (1,078 rows)
split 2: train up to 2019-01-12 (2,157 rows), validate from 2019-01-12 to 2020-10-03 (1,078 rows)
split 3: train up to 2020-10-03 (3,235 rows), validate from 2020-10-03 to 2022-03-05 (1,078 rows)
split 4: train up to 2022-03-05 (4,313 rows), validate from 2022-03-05 to 2023-09-25 (1,078 rows)
split 5: train up to 2023-09-25 (5,391 rows), validate from 2023-09-25 to 2025-02-15 (1,078 rows)


In [5]:
# the shared tools in preprocessing.py:
#   model_columns  the columns the model sees (no target, customer or date)
#   prepare_base   median / mode, fitted on train only
#   fit_predict    prepare, encode, scale and train on train; predict test
#   evaluate       F1 and accuracy on each split 
#   compare        each alternative against a reference, split by split
from preprocessing import (as_object, compare, evaluate, fit_predict, model_columns,
                           prepare_base, sem)

In [6]:
# baseline: median / mode, one-hot, StandardScaler, no new features
baseline = evaluate(prepare_base, frame, splits)

print(f"baseline: F1 {baseline['f1'].mean():.4f} +- {sem(baseline['f1']):.4f}"
      f" | accuracy {baseline['accuracy'].mean():.4f} +- {sem(baseline['accuracy']):.4f}")
baseline.round(4)

baseline: F1 0.7321 +- 0.0090 | accuracy 0.7056 +- 0.0035


,f1,accuracy
0,0.7543,0.6929
1,0.7258,0.7106
2,0.7350,0.7124
3,0.7015,0.7032
4,0.7439,0.7087


In [7]:
# the floor: always predict 1 (the majority class) on the same validation rows
always_one = pd.DataFrame({
    "f1": [f1_score(frame[TARGET].iloc[v], np.ones(len(v), dtype=int)) for _, v in splits],
    "accuracy": [frame[TARGET].iloc[v].mean() for _, v in splits],
})

pd.DataFrame({"baseline": baseline.mean(), "always predict 1": always_one.mean()}).round(4)

,baseline,always predict 1
f1,0.7321,0.7206
accuracy,0.7056,0.5638


## 2. Impossible values

The EDA found values that cannot be true: negative day counts, a placeholder of 1e9, and
means larger than the maximum of the same purchases. The rules are in `preprocessing.py`
(`invalid_errors` and `invalid_flags`). We compare setting those cells to null with dropping
the training rows that carry them.

We set them to null with `blank_invalid`. Dropping the rows ties (F1 +0.0006, noise), and
setting to null is the only option that also works on `test.csv`, where every row needs a
prediction. The rule learns nothing from the data, so it is applied the same way to train,
validation and test.

In [8]:
from preprocessing import blank_invalid, invalid_flags

flags = invalid_flags(frame)
print("impossible values per column:", flags.sum().to_dict())
print("rows with an impossible value:", int(flags.any(axis=1).sum()), "of", len(frame))

impossible values per column: {'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16, 'prior_mean_purchase_value': 77, 'spend_365d': 30}
rows with an impossible value: 139 of 6469


In [9]:
# keep, set to null, or drop the row? In every case the remaining nulls get the median / mode
def prepare_blank(train, test):
    """Impossible values become null, in both halves."""
    return prepare_base(blank_invalid(train), blank_invalid(test))


def prepare_drop(train, test):
    """Drop the TRAINING rows with impossible values.

    Validation never loses rows, because every test purchase needs a prediction.
    """
    keep = ~invalid_flags(train).any(axis=1)
    return prepare_base(train[keep], blank_invalid(test))


invalid_runs = {"set to null": evaluate(prepare_blank, frame, splits),
                "drop the row (training)": evaluate(prepare_drop, frame, splits)}
compare(invalid_runs, baseline).round(4)

,f1,accuracy,vs base,SEM,better on
set to null,0.734164,0.707607,0.00206,0.0011,4 of 5
drop the row (training),0.734751,0.707421,0.002646,0.001175,5 of 5


In [10]:
print("test rows without a date:", int(test_csv[DATE].isna().sum()))
print("impossible values in the test set (the rule applies the same way):",
      int(invalid_flags(test_csv).sum().sum()))

test rows without a date: 0
impossible values in the test set (the rule applies the same way): 18


## 3. Missing values

Most of the missing values are structural: a history column cannot exist yet when the
customer has too few previous purchases (a mean gap needs two purchases, the spread of the
gaps needs three). We compare three fill plans: median and mode, zero where the value cannot
exist with the median elsewhere, and a level of its own for missing categories.

All three tie, so we keep the simplest: the median for numeric columns and the mode for
categorical ones, fitted on the training rows only.

In [11]:
# nulls left after the impossible values are blanked
blanked = blank_invalid(frame)
nulls = blanked.isna().sum()
nulls[nulls > 0].sort_values(ascending=False).to_frame("nulls")

,nulls
prior_std_gap_days,1413
prior_mean_gap_days,1259
prior_std_purchase_value,1048
prior_mean_purchase_value,874
days_since_previous_purchase,812
prior_min_purchase_value,603
prior_max_purchase_value,603
spend_365d,225
transport_method,196
payment_type,195


In [12]:
# minimum number of previous purchases for each history column to exist
needs_history = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1,
    "prior_min_purchase_value": 1,
    "prior_max_purchase_value": 1,
    "prior_std_purchase_value": 2,   # no standard deviation of a single value
    "prior_mean_gap_days": 2,        # a gap needs two purchases
    "prior_std_gap_days": 3,         # the spread of gaps needs two gaps
}


def zero_where_none(frame):
    """Zero where the value cannot exist yet; the other nulls are left for the median.

    It learns nothing from
    the data, so it is applied the same way to train and test.
    """
    out = frame.copy()
    for column, minimum in needs_history.items():
        no_history = out[column].isna() & (out["prior_purchase_count"] < minimum)
        out.loc[no_history, column] = 0
    return out


def own_level(frame):
    """Nulls in categorical columns become a level of their own, "(missing)"."""
    out = frame.copy()
    _, categorical = model_columns(out)
    out[categorical] = out[categorical].fillna("(missing)")
    return out


structural = int(blanked.isna().sum().sum() - zero_where_none(blanked).isna().sum().sum())
print(f"nulls in total: {int(blanked.isna().sum().sum()):,}, of which structural: {structural:,}")

nulls in total: 7,617, of which structural: 5,921


In [13]:
# the fill plans, all after the impossible values are blanked
def prepare_zero_where_none(train, test):
    return prepare_base(zero_where_none(blank_invalid(train)), zero_where_none(blank_invalid(test)))


def prepare_own_level(train, test):
    return prepare_base(own_level(blank_invalid(train)), own_level(blank_invalid(test)))


fill_runs = {"median / mode": invalid_runs["set to null"],
             "zero where none / median elsewhere": evaluate(prepare_zero_where_none, frame, splits),
             "median / own level": evaluate(prepare_own_level, frame, splits)}
compare(fill_runs, invalid_runs["set to null"]).round(4)

,f1,accuracy,vs base,SEM,better on
median / mode,0.734164,0.707607,0.0,0.0,0 of 5
zero where none / median elsewhere,0.734237,0.707421,0.000074,0.000404,2 of 5
median / own level,0.734468,0.707607,0.000304,0.001256,2 of 5


In [14]:
# sections 2 and 3 in one preparation, the reference (`clean`) for the next sections
prepare_clean = prepare_blank

clean = invalid_runs["set to null"]
print(f"impossible values + missing values: F1 {clean['f1'].mean():.4f} +- {sem(clean['f1']):.4f}"
      f" | accuracy {clean['accuracy'].mean():.4f} +- {sem(clean['accuracy']):.4f}")

impossible values + missing values: F1 0.7342 +- 0.0093 | accuracy 0.7076 +- 0.0024


## 4. Long tails and scaling

27 of the 29 numeric columns have a long right tail (all but `random_noise` and
`customer_tenure_days`). We compare two treatments, `log1p` and winsorizing at the 1% and
99% training quantiles, and then four scaling options. Everything is compared against
`clean`, and then each change is measured on its own.

We keep `log1p` on the 27 long-tailed columns followed by `MinMaxScaler`. Together they
reach F1 0.7481, +0.0139 against `clean` (2 × SEM = 0.0056) and better on all 5 splits, the
only gain in this section clearly above the noise. Measured separately, MinMax beats
Standard (both with log1p) by +0.0084 in F1 and +0.0085 in accuracy, and log1p beats no
log1p (both with MinMax) by +0.0073 in F1. `log1p` learns nothing from the data, so it
cannot leak. `RobustScaler` also passes the noise but scores lower, and without a scaler the
model does not converge.

In [15]:
# the long-tailed numeric columns: all but random_noise and customer_tenure_days
skewed = [c for c in model_columns(frame)[0] if c not in ["random_noise", "customer_tenure_days"]]
from preprocessing import SKEWED
assert skewed == SKEWED   # the same list preprocessing.py uses for log1p
print(f"columns to transform: {len(skewed)} of {len(model_columns(frame)[0])} numeric")

# skew before and after log1p, only to see the effect: nothing is fitted
blanked = blank_invalid(frame)
pd.DataFrame({"skew before": blanked[skewed].skew(),
              "skew after log1p": np.log1p(blanked[skewed]).skew()}).round(2)

columns to transform: 27 of 29 numeric


,skew before,skew after log1p
Total a Pagar,4.21,-0.48
Total Bruto,4.31,-0.43
Total Liquido,4.15,-0.45
Total Imp_,4.84,-0.76
Total Portes,19.55,2.27
Total Desc_ Global,27.46,9.28
Total Desc_ Linha,5.92,0.24
Custo,4.61,-1.02
N_ Detalhes,2.98,0.08
invoice_count_current_purchase,6.44,5.00


In [16]:
# order: impossible values -> treatment -> nulls. After blank_invalid there are no negatives
# left in these columns, so log1p does not need a clip(lower=0) first
from preprocessing import prepare_log1p


def prepare_winsor(train, test):
    """Clip each column to the 1% and 99% quantiles of TRAIN.

    The limits are learned on `train` only and applied the same way to `test`.
    """
    train, test = blank_invalid(train), blank_invalid(test)
    low, high = train[skewed].quantile(0.01), train[skewed].quantile(0.99)
    train[skewed] = train[skewed].clip(low, high, axis=1)
    test[skewed] = test[skewed].clip(low, high, axis=1)
    return prepare_base(train, test)

In [17]:
# the treatments against `clean`, with the same StandardScaler
treatment_runs = {"log1p": evaluate(prepare_log1p, frame, splits),
                  "winsorize 1% / 99%": evaluate(prepare_winsor, frame, splits)}
treatment_table = compare(treatment_runs, clean)
treatment_table["2 x SEM"] = 2 * treatment_table["SEM"]
treatment_table.round(4)

,f1,accuracy,vs base,SEM,better on,2 x SEM
log1p,0.739642,0.707236,0.005478,0.002904,4 of 5,0.005808
winsorize 1% / 99%,0.737703,0.709647,0.003539,0.002745,3 of 5,0.005491


In [18]:
# the scalers on top of log1p; the reference is still `clean`
scaler_runs = {"log1p + StandardScaler": treatment_runs["log1p"],
               "log1p + MinMaxScaler": evaluate(prepare_log1p, frame, splits, scaler=MinMaxScaler()),
               "log1p + RobustScaler": evaluate(prepare_log1p, frame, splits, scaler=RobustScaler()),
               "log1p, unscaled": evaluate(prepare_log1p, frame, splits, scaler=None)}
scaler_table = compare(scaler_runs, clean)
scaler_table["2 x SEM"] = 2 * scaler_table["SEM"]
scaler_table.round(4)

,f1,accuracy,vs base,SEM,better on,2 x SEM
log1p + StandardScaler,0.739642,0.707236,0.005478,0.002904,4 of 5,0.005808
log1p + MinMaxScaler,0.748076,0.71577,0.013912,0.002819,5 of 5,0.005637
log1p + RobustScaler,0.743428,0.713173,0.009264,0.002626,5 of 5,0.005251
"log1p, unscaled",0.742002,0.708905,0.007838,0.004524,4 of 5,0.009048


In [19]:
# each change measured on its own, on the same splits
clean_minmax = evaluate(prepare_clean, frame, splits, scaler=MinMaxScaler())
for metric in ["f1", "accuracy"]:
    pair_table = pd.concat([
        compare({"MinMax vs Standard (both with log1p)": scaler_runs["log1p + MinMaxScaler"]},
                treatment_runs["log1p"], metric=metric),
        compare({"log1p vs no log1p (both with MinMax)": scaler_runs["log1p + MinMaxScaler"]},
                clean_minmax, metric=metric),
    ])
    pair_table["2 x SEM"] = 2 * pair_table["SEM"]
    print(f"[{metric}]")
    print(pair_table[[metric, "vs base", "SEM", "2 x SEM", "better on"]].round(4).to_string(), "\n")

[f1]
                                            f1   vs base       SEM   2 x SEM better on
MinMax vs Standard (both with log1p)  0.748076  0.008435  0.003569  0.007138    4 of 5
log1p vs no log1p (both with MinMax)  0.748076  0.007298  0.001651  0.003302    5 of 5 

[accuracy]
                                     accuracy   vs base      SEM  2 x SEM better on
MinMax vs Standard (both with log1p)  0.71577  0.008534  0.00335   0.0067    4 of 5
log1p vs no log1p (both with MinMax)  0.71577   0.00538  0.00291  0.00582    3 of 5 



In [20]:
# the scaler is not part of `prepare`: it is passed to evaluate / fit_predict
SCALER_CHOSEN = MinMaxScaler()

log1p_minmax = scaler_runs["log1p + MinMaxScaler"]
print(f"log1p + MinMaxScaler: F1 {log1p_minmax['f1'].mean():.4f} +- {sem(log1p_minmax['f1']):.4f}"
      f" | accuracy {log1p_minmax['accuracy'].mean():.4f} +- {sem(log1p_minmax['accuracy']):.4f}")

log1p + MinMaxScaler: F1 0.7481 +- 0.0086 | accuracy 0.7158 +- 0.0052


## 5. New features

We built five candidate features from what is known at the moment of the purchase: a
first-purchase flag (in the EDA only 33% of first purchases were repeated, against 58% of
the others), how late the purchase is compared with the customer's usual gap, the average
ticket over the last year, the margin, and the share of the discount. Each one is added on
its own to `clean`, then all five together, and as a check the same features are added on
top of log1p and MinMaxScaler.

None of them is kept. None passes 2 × SEM (the closest, `discount_share`, gains +0.0019 with
2 × SEM = 0.0020), the five together stay in the noise (+0.0024), and none helps on top of
log1p and MinMaxScaler. The information is already in the existing columns:
`is_first_purchase`, for example, is just `prior_purchase_count == 0`.

In [21]:
# the candidates; a zero denominator becomes null (.replace(0, np.nan)),
# later filled with the training median by prepare_base
new_features = {
    # a first purchase behaves differently (33% against 58% repeat rate in the EDA)
    "is_first_purchase": lambda t: (t["prior_purchase_count"] == 0).astype(int),
    # delay compared with the customer's usual gap
    "days_vs_habit": lambda t: t["days_since_previous_purchase"]
                               / t["prior_mean_gap_days"].replace(0, np.nan),
    # average value per purchase over the last year
    "avg_ticket_365d": lambda t: t["spend_365d"] / t["purchase_count_365d"].replace(0, np.nan),
    # margin of the purchase
    "margin": lambda t: t["Total Liquido"] - t["Custo"],
    # weight of the discount in the purchase
    "discount_share": lambda t: t["Total Desc_ Linha"] / t["Total Bruto"].replace(0, np.nan),
}


def add_features(frame, names=tuple(new_features)):
    """Add the features `names` to a table. It learns nothing from the data."""
    out = frame.copy()
    for name in names:
        out[name] = new_features[name](out)
    return out


with_features = add_features(blank_invalid(frame))
with_features[list(new_features)].describe().T[["count", "min", "50%", "max"]].round(3)

,count,min,50%,max
is_first_purchase,6469.0,0.000,0.000,1.000
days_vs_habit,5031.0,0.001,0.831,478.667
avg_ticket_365d,5434.0,3.310,720.128,66539.157
margin,6469.0,-1592.119,197.205,11075.600
discount_share,6469.0,0.000,0.046,0.975


In [22]:
# order: blank_invalid -> add_features -> prepare_base, so features never use filled values
def prepare_with(names):
    """Return prepare(train, test) = prepare_clean + the features `names`."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        return prepare_base(train, test)
    return prepare

In [23]:
# each feature on its own against `clean`, then all five together
feature_runs = {name: evaluate(prepare_with([name]), frame, splits) for name in new_features}
feature_runs["all 5"] = evaluate(prepare_with(list(new_features)), frame, splits)
feature_table = compare(feature_runs, clean)
feature_table["2 x SEM"] = 2 * feature_table["SEM"]
feature_table.round(4)

,f1,accuracy,vs base,SEM,better on,2 x SEM
is_first_purchase,0.734286,0.707607,0.000122,0.00075,1 of 5,0.0015
days_vs_habit,0.734271,0.707421,0.000107,0.00069,2 of 5,0.001379
avg_ticket_365d,0.733388,0.706865,-0.000776,0.000222,0 of 5,0.000444
margin,0.734217,0.707792,0.000053,0.000306,2 of 5,0.000611
discount_share,0.736052,0.709091,0.001888,0.001013,4 of 5,0.002027
all 5,0.736541,0.708349,0.002378,0.001351,4 of 5,0.002701


In [24]:
# check: the same features on top of log1p + MinMaxScaler, against `log1p_minmax`
def prepare_with_log1p(names):
    """prepare_log1p + the features `names`; log1p only touches the original columns (`skewed`)."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        train[skewed], test[skewed] = np.log1p(train[skewed]), np.log1p(test[skewed])
        return prepare_base(train, test)
    return prepare


feature_runs_log1p = {name: evaluate(prepare_with_log1p([name]), frame, splits, scaler=SCALER_CHOSEN)
                   for name in new_features}
feature_table_log1p = compare(feature_runs_log1p, log1p_minmax)
feature_table_log1p["2 x SEM"] = 2 * feature_table_log1p["SEM"]
feature_table_log1p.round(4)

,f1,accuracy,vs base,SEM,better on,2 x SEM
is_first_purchase,0.746513,0.714286,-0.001563,0.000773,1 of 5,0.001546
days_vs_habit,0.748472,0.716327,0.000396,0.000597,2 of 5,0.001194
avg_ticket_365d,0.748189,0.715955,0.000113,0.000489,2 of 5,0.000978
margin,0.748568,0.716512,0.000492,0.000282,3 of 5,0.000565
discount_share,0.747299,0.715213,-0.000777,0.00083,2 of 5,0.00166


## 6. Categorical columns

We do not merge rare levels into an "other" level. We test instead: `handle_unknown` in the
encoder for levels that only appear in validation, compact encodings (count and target), and
leaving out a column with too many levels. Target encoding uses the target, so it is fitted
on each training split only.

One-hot stays, as in the baseline. Against `clean` the ordinal encoder is the only one above
the noise (F1 +0.0072), but on top of log1p and MinMaxScaler, the preparation that goes into
the final integration, its advantage disappears (+0.0008, 2 × SEM = 0.0107). It also imposes
an order the levels do not have. `transport_method` stays for now: without it F1 stays in
the noise, but accuracy improves (+0.0085 on top of log1p and MinMax, just above the noise),
so feature selection looks at it again.

In [25]:
# what there is to encode
_, categorical = model_columns(frame)
pd.DataFrame({
    "levels": frame[categorical].nunique(),
    "levels with fewer than 30 rows": [int((frame[c].value_counts() < 30).sum())
                                       for c in categorical],
    "nulls": frame[categorical].isna().sum(),
}).sort_values("levels", ascending=False)

,levels,levels with fewer than 30 rows,nulls
transport_method,80,51,196
payment_type,9,4,195
salesperson,9,3,194
commercial_zone,5,2,0
document_series,3,0,0
warehouse,2,1,195


In [26]:
# per split: how many columns the one-hot creates and how many validation rows
# carry a level the training rows never saw, which is why handle_unknown exists
width_rows = []
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train, test = prepare_clean(frame.iloc[train_idx], frame.iloc[val_idx])
    unseen = pd.concat([~test[c].isin(train[c]) for c in categorical], axis=1)
    width_rows.append({
        "split": fold,
        "one-hot columns": OneHotEncoder(handle_unknown="ignore")
                          .fit(as_object(train[categorical])).get_feature_names_out().size,
        "of which transport_method": train["transport_method"].nunique(),
        "validation rows with an unseen level": int(unseen.any(axis=1).sum()),
        "in transport_method": int(unseen["transport_method"].sum()),
    })
pd.DataFrame(width_rows).set_index("split")

,one-hot columns,of which transport_method,validation rows with an unseen level,in transport_method
split,,,,
1,56,42,178,13
2,67,51,132,132
3,80,62,634,219
4,94,71,181,22
5,104,76,27,27


In [27]:
# four encoders on top of prepare_clean, against `clean`;
# with drop="first" an unseen level is all zeros, like the dropped level, and scikit-learn
# warns about it on every split, so the warning is silenced
warnings.filterwarnings("ignore", category=UserWarning)

encoders = {
    "one-hot, drop='first'": OneHotEncoder(handle_unknown="ignore", drop="first",
                                           sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "count": CountEncoder(normalize=True, handle_unknown=0, handle_missing="value"),
    "target": TargetEncoder(handle_unknown="value", handle_missing="value"),
}


def prepare_without_transport(train, test):
    """prepare_clean without transport_method, the column with 80 levels."""
    return prepare_clean(train.drop(columns="transport_method"),
                         test.drop(columns="transport_method"))


encoding_runs = {name: evaluate(prepare_clean, frame, splits, encoder=encoder)
                 for name, encoder in encoders.items()}
encoding_runs["one-hot, without transport_method"] = evaluate(prepare_without_transport,
                                                              frame, splits)
for metric in ["f1", "accuracy"]:
    encoding_table = compare(encoding_runs, clean, metric=metric)
    encoding_table["2 x SEM"] = 2 * encoding_table["SEM"]
    print(f"[{metric}]")
    print(encoding_table.round(4).to_string(), "\n")

[f1]
                                         f1  accuracy   vs base       SEM better on   2 x SEM
one-hot, drop='first'              0.734485  0.708163  0.000321  0.001204    2 of 5  0.002407
ordinal                            0.741409  0.718738  0.007245  0.002307    5 of 5  0.004614
count                              0.730803  0.712987 -0.003361  0.006088    3 of 5  0.012177
target                             0.733991  0.712801 -0.000173  0.003446    2 of 5  0.006893
one-hot, without transport_method  0.735859  0.717069  0.001695   0.00437    3 of 5  0.008741 

[accuracy]
                                   accuracy   vs base       SEM better on   2 x SEM
one-hot, drop='first'              0.708163  0.000557  0.001775    3 of 5  0.003549
ordinal                            0.718738  0.011132   0.00231    5 of 5   0.00462
count                              0.712987   0.00538  0.003096    4 of 5  0.006192
target                             0.712801  0.005195  0.002761    4 of 5  0.00552

In [28]:
# check: the same encodings on top of log1p + MinMaxScaler, against `log1p_minmax`
def prepare_log1p_without_transport(train, test):
    """prepare_log1p without transport_method."""
    return prepare_log1p(train.drop(columns="transport_method"),
                      test.drop(columns="transport_method"))


encoding_runs_log1p = {name: evaluate(prepare_log1p, frame, splits, scaler=SCALER_CHOSEN, encoder=encoder)
                    for name, encoder in encoders.items()}
encoding_runs_log1p["one-hot, without transport_method"] = evaluate(
    prepare_log1p_without_transport, frame, splits, scaler=SCALER_CHOSEN)
for metric in ["f1", "accuracy"]:
    encoding_table_log1p = compare(encoding_runs_log1p, log1p_minmax, metric=metric)
    encoding_table_log1p["2 x SEM"] = 2 * encoding_table_log1p["SEM"]
    print(f"[{metric}]")
    print(encoding_table_log1p.round(4).to_string(), "\n")

[f1]
                                         f1  accuracy   vs base       SEM better on   2 x SEM
one-hot, drop='first'              0.746571  0.716141 -0.001505   0.00293    2 of 5   0.00586
ordinal                            0.748861  0.717254  0.000785  0.005354    3 of 5  0.010708
count                              0.743285  0.718553 -0.004791  0.005319    3 of 5  0.010639
target                             0.749137  0.719852  0.001061  0.003108    2 of 5  0.006216
one-hot, without transport_method  0.752546  0.724304   0.00447  0.003436    4 of 5  0.006872 

[accuracy]
                                   accuracy   vs base       SEM better on   2 x SEM
one-hot, drop='first'              0.716141  0.000371  0.002465    2 of 5   0.00493
ordinal                            0.717254  0.001484  0.004644    4 of 5  0.009288
count                              0.718553  0.002783  0.001901    4 of 5  0.003802
target                             0.719852  0.004082  0.004781    1 of 5  0.00956

In [29]:
ENCODER_CHOSEN = None   # None = the one-hot of fit_predict

## 7. The optional `clientes.csv` table

Only three of the 12 columns of `clientes.csv` are usable: `customer_since_year`,
`country_or_region` and `commercial_region`. The others are empty, identify the customer,
have too many levels, or are missing for almost half the customers. Since `customer_id` is
the index of the table, the join is a lookup with `.loc`; it learns nothing from the data,
so it is applied the same way to train, validation and test. Each column is tested on its
own and the three together, against `clean` and on top of log1p and MinMaxScaler.

We do not use `clientes.csv`. The join itself works: every customer is in the table,
`customer_since_year` is never later than the purchase, and its correlation with
`customer_tenure_days` is only 0.32. But none of the three columns helps.
`customer_since_year` lowers F1 (−0.0068, 2 × SEM = 0.0035, better on 0 of 5), the other two
stay in the noise, and the three together are worse (−0.0087). What the table says about a
customer is already in the purchases.

In [30]:
# read clientes.csv as in the EDA; levels and nulls only count the 610 training customers
clientes = pd.read_csv("../data/clientes.csv", index_col="customer_id")
client_columns = ["customer_since_year", "country_or_region", "commercial_region"]

print(f"{len(clientes):,} customers, {clientes.index.nunique():,} distinct customer_id")
print("training customers in clientes.csv:", raw[GROUP].isin(clientes.index).all(),
      "| test.csv customers in clientes.csv:", test_csv[GROUP].isin(clientes.index).all())

in_train = clientes[clientes.index.isin(raw[GROUP])]
reasons = {
    "customer_type": "left out: empty",
    "company_name": "left out: identifies the customer",
    "address": "left out: identifies the customer",
    "island": "left out: missing for almost half the customers",
}
for column in ["postal_code", "street", "city", "postal_area", "municipality"]:
    reasons[column] = "left out: too many levels"
for column in client_columns:
    reasons[column] = "candidate"
pd.DataFrame({"% nulls": (100 * in_train.isna().mean()).round(1),
              "levels": in_train.nunique(),
              "decision": pd.Series(reasons)})

1,143 customers, 1,143 distinct customer_id
training customers in clientes.csv: True | test.csv customers in clientes.csv: True


,% nulls,levels,decision
address,0.0,600,left out: identifies the customer
city,38.7,260,left out: too many levels
commercial_region,2.0,32,candidate
company_name,0.0,609,left out: identifies the customer
country_or_region,0.0,7,candidate
customer_since_year,0.0,10,candidate
customer_type,100.0,0,left out: empty
island,46.9,10,left out: missing for almost half the customers
municipality,6.7,150,left out: too many levels
postal_area,6.6,189,left out: too many levels


In [31]:
# lookup by index (customer_id is the index of clientes), the same for every part of the data
def add_client_columns(frame, columns):
    """Add to each purchase the `columns` of its customer, read from clientes.csv."""
    out = frame.copy()
    for column in columns:   # one column at a time, so each keeps its own type
        out[column] = clientes.loc[out[GROUP], column].to_numpy()
    return out


joined = add_client_columns(frame, client_columns)
print("rows before and after the join:", len(frame), len(joined))
print("nulls after the join:", joined[client_columns].isna().sum().to_dict())

# customer_since_year cannot be later than the purchase (that would be future information);
# dates compare as text, as in the EDA: "2018" sorts before "2018-05-01" and after "2017-12-31"
print("purchases with customer_since_year after the purchase:",
      int((joined["customer_since_year"].astype(str) > joined[DATE]).sum()))
# does it repeat customer_tenure_days? (redundant above 0.70, as in the EDA)
print(f"|r| between customer_since_year and customer_tenure_days:"
      f" {abs(joined['customer_since_year'].corr(joined['customer_tenure_days'])):.2f}")

rows before and after the join: 6469 6469
nulls after the join: {'customer_since_year': 0, 'country_or_region': 0, 'commercial_region': 35}
purchases with customer_since_year after the purchase: 0
|r| between customer_since_year and customer_tenure_days: 0.32


In [32]:
# each column on its own and the three together, on top of prepare_clean, against `clean`
def prepare_with_clients(columns, prepare=prepare_clean):
    """Return prepare(train, test) = `prepare` + the clientes columns `columns`."""
    def prepare_joined(train, test):
        return prepare(add_client_columns(train, columns), add_client_columns(test, columns))
    return prepare_joined


client_runs = {column: evaluate(prepare_with_clients([column]), frame, splits)
               for column in client_columns}
client_runs["all 3 together"] = evaluate(prepare_with_clients(client_columns), frame, splits)
for metric in ["f1", "accuracy"]:
    client_table = compare(client_runs, clean, metric=metric)
    client_table["2 x SEM"] = 2 * client_table["SEM"]
    print(f"[{metric}]")
    print(client_table.round(4).to_string(), "\n")

[f1]
                           f1  accuracy   vs base       SEM better on   2 x SEM
customer_since_year  0.727414  0.708163  -0.00675  0.001731    0 of 5  0.003461
country_or_region    0.733152  0.706308 -0.001012  0.000627    0 of 5  0.001255
commercial_region    0.732812  0.705566 -0.001352  0.001278    1 of 5  0.002557
all 3 together       0.725434  0.705751  -0.00873  0.001999    0 of 5  0.003998 

[accuracy]
                     accuracy   vs base       SEM better on   2 x SEM
customer_since_year  0.708163  0.000557  0.003221    1 of 5  0.006443
country_or_region    0.706308 -0.001299   0.00086    0 of 5  0.001721
commercial_region    0.705566 -0.002041  0.001188    1 of 5  0.002376
all 3 together       0.705751 -0.001855  0.002705    2 of 5  0.005409 



In [33]:
# check: the same columns on top of log1p + MinMaxScaler, against `log1p_minmax`;
# log1p only touches the original columns, so customer_since_year is not transformed
client_runs_log1p = {column: evaluate(prepare_with_clients([column], prepare_log1p), frame, splits,
                                   scaler=SCALER_CHOSEN)
                  for column in client_columns}
client_runs_log1p["all 3 together"] = evaluate(prepare_with_clients(client_columns, prepare_log1p),
                                            frame, splits, scaler=SCALER_CHOSEN)
for metric in ["f1", "accuracy"]:
    client_table_log1p = compare(client_runs_log1p, log1p_minmax, metric=metric)
    client_table_log1p["2 x SEM"] = 2 * client_table_log1p["SEM"]
    print(f"[{metric}]")
    print(client_table_log1p.round(4).to_string(), "\n")

[f1]
                           f1  accuracy   vs base       SEM better on   2 x SEM
customer_since_year  0.744235  0.717996 -0.003841   0.00245    2 of 5  0.004899
country_or_region    0.747492  0.715584 -0.000584   0.00068    2 of 5  0.001361
commercial_region    0.745636  0.712801 -0.002441  0.002283    2 of 5  0.004566
all 3 together       0.741504  0.715955 -0.006572  0.003175    1 of 5  0.006349 

[accuracy]
                     accuracy   vs base       SEM better on   2 x SEM
customer_since_year  0.717996  0.002226   0.00175    2 of 5  0.003501
country_or_region    0.715584 -0.000186  0.000899    3 of 5  0.001799
commercial_region    0.712801 -0.002968  0.003192    2 of 5  0.006384
all 3 together       0.715955  0.000186  0.002203    2 of 5  0.004406 



## 8. Final preparation

Only sections 2, 3 and 4 change the preparation; the other sections kept the one-hot, left
out `clientes.csv` and added no features. `preprocessing.py` saves the result as
`prepare_final` (impossible values set to null, log1p, median and mode), `ENCODER_FINAL`
(one-hot) and `SCALER_FINAL` (`MinMaxScaler`). For `test.csv` it is fitted on the whole
training set and applied unchanged.

The final preparation reaches F1 0.7481 and accuracy 0.7158, against 0.7321 and 0.7056 for
the baseline.

In [34]:
from preprocessing import ENCODER_FINAL, SCALER_FINAL, prepare_final
assert prepare_final is prepare_log1p      # impossible values -> null, log1p, median / mode
assert ENCODER_FINAL is ENCODER_CHOSEN      # None = one-hot with handle_unknown="ignore"
assert type(SCALER_FINAL) is type(SCALER_CHOSEN)   # MinMaxScaler

final_run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
for metric in ["f1", "accuracy"]:
    final_table = compare({"sections 2 + 3 (clean)": clean, "final (sections 2 + 3 + 4)": final_run},
                          baseline, metric=metric)
    final_table["2 x SEM"] = 2 * final_table["SEM"]
    print(f"[{metric}] against the baseline")
    print(final_table[[metric, "vs base", "SEM", "2 x SEM", "better on"]].round(4).to_string(), "\n")

[f1] against the baseline
                                  f1   vs base       SEM   2 x SEM better on
sections 2 + 3 (clean)      0.734164   0.00206    0.0011    0.0022    4 of 5
final (sections 2 + 3 + 4)  0.748076  0.015972  0.001742  0.003484    5 of 5 

[accuracy] against the baseline
                            accuracy   vs base       SEM   2 x SEM better on
sections 2 + 3 (clean)      0.707607  0.002041  0.001644  0.003288    3 of 5
final (sections 2 + 3 + 4)   0.71577  0.010204  0.001968  0.003936    5 of 5 



In [35]:
# the final preparation on test.csv: fitted on the whole training set, applied unchanged
train_final, test_final = prepare_final(frame, test_csv)
numeric, categorical = model_columns(train_final)

pd.Series({
    "rows: train / test": f"{len(train_final):,} / {len(test_final):,}",
    "same columns, in the same order": list(model_columns(test_final)[0]) == numeric
                                       and list(model_columns(test_final)[1]) == categorical,
    "nulls in train / test": f"{train_final[numeric + categorical].isna().sum().sum()}"
                             f" / {test_final[numeric + categorical].isna().sum().sum()}",
    "impossible values in test: before / after blank_invalid":
        f"{invalid_flags(test_csv).sum().sum()} / {invalid_flags(blank_invalid(test_csv)).sum().sum()}",
    # levels the training set does not have: the one-hot sets them to zeros
    "test rows with an unseen level, per column":
        {c: int((~test_final[c].isin(train_final[c])).sum()) for c in categorical},
}).to_frame("check")

,check
rows: train / test,"6,469 / 986"
"same columns, in the same order",True
nulls in train / test,0 / 0
impossible values in test: before / after blank_invalid,18 / 0
"test rows with an unseen level, per column","{'payment_type': 3, 'salesperson': 0, 'commercial_zone': 209, 'transport_method': 63, 'warehouse': 7, 'document_series': 0}"


In [36]:
# every preprocessing decision in one table; "noise" = below 2 x SEM
decisions = pd.DataFrame([
    ("1. Starting point", "time-based splits, TimeSeriesSplit(5)", "by customer (GroupKFold)",
     "test.csv is later than the training data and 94.5% of its rows are known customers"),
    ("1. Starting point", "65 rows without a date are left out", "-",
     "they only hold customer and target; test.csv has no such rows"),
    ("2. Impossible values", "impossible values -> null", "keep; drop the row",
     "ties with dropping (F1 +0.0006, noise) and is the only option that works on test.csv"),
    ("3. Missing values", "nulls -> median / mode", "zero where none; own level",
     "all tie (noise); the simplest stays"),
    ("6. Categorical columns", "one-hot, rare levels not merged", "drop='first', ordinal, count, target",
     "on top of log1p + MinMax none passes 2 x SEM; ordinal imposes an order"),
    ("6. Categorical columns", "transport_method stays (for now)", "remove it",
     "F1 in the noise; accuracy +0.0085 (2 x SEM 0.0082): decided in feature selection"),
    ("4. Long tails and scaling", "log1p on the 27 long-tailed columns", "keep; winsorize 1% / 99%",
     "with MinMax: F1 +0.0073 (2 x SEM 0.0033); no row is removed"),
    ("4. Long tails and scaling", "MinMaxScaler", "Standard, Robust, unscaled",
     "with log1p: F1 +0.0084, accuracy +0.0085 against Standard; unscaled does not converge"),
    ("7. clientes.csv", "clientes.csv is not used", "3 columns, alone and together",
     "none helps; customer_since_year lowers F1 (-0.0068, 0 of 5)"),
    ("5. New features", "no new feature", "5 features, alone and together",
     "none passes 2 x SEM: the information is already in the existing columns"),
], columns=["section", "decision", "alternatives tested", "why"])
print(decisions.to_string(index=False))

                  section                              decision                  alternatives tested                                                                                   why
        1. Starting point time-based splits, TimeSeriesSplit(5)             by customer (GroupKFold)    test.csv is later than the training data and 94.5% of its rows are known customers
        1. Starting point   65 rows without a date are left out                                    -                         they only hold customer and target; test.csv has no such rows
     2. Impossible values             impossible values -> null                   keep; drop the row  ties with dropping (F1 +0.0006, noise) and is the only option that works on test.csv
        3. Missing values                nulls -> median / mode           zero where none; own level                                                   all tie (noise); the simplest stays
   6. Categorical columns       one-hot, rare levels not merged d

## 9. Feature selection: identifiers and the reference

The homework asks for identifier-like variables to be treated critically. `ID`,
`customer_id` and `purchase_date` do not describe the purchase, so none of them enters the
selection. As a number, `customer_id` would make customer 900000 "larger" than customer
100000, and the time information is already in the recency, tenure and window columns.

The reference for feature selection is the final preparation with every column. After the one-hot,
the model sees 85 to 133 columns depending on the split, and the selectors below remove
columns from that set. All the selection methods are the ones covered in class, and each
selector is fitted on the training rows of each split only.

In [37]:
identifier_like = pd.DataFrame([
    ("ID", "table index", "only identifies the row (and the Kaggle submission)"),
    ("customer_id", "OUT_OF_MODEL", "identifies the customer; as a number, the model would read"
                                    " customer 900000 as 'larger' than 100000"),
    ("purchase_date", "OUT_OF_MODEL", "text; it orders the splits, and the time information is"
                                      " already in recency, tenure and the windows"),
], columns=["column", "where it is", "why it is not a feature"])
print(identifier_like.to_string(index=False))

# check that none of them reaches the model
numeric, categorical = model_columns(frame)
print("\nidentifiers among the model columns:",
      [c for c in ["ID", GROUP, DATE] if c in numeric + categorical])
print(f"columns before the one-hot: {len(numeric)} numeric and {len(categorical)} categorical")

       column  where it is                                                                            why it is not a feature
           ID  table index                                                only identifies the row (and the Kaggle submission)
  customer_id OUT_OF_MODEL identifies the customer; as a number, the model would read customer 900000 as 'larger' than 100000
purchase_date OUT_OF_MODEL text; it orders the splits, and the time information is already in recency, tenure and the windows

identifiers among the model columns: []
columns before the one-hot: 29 numeric and 6 categorical


In [38]:
# the feature selection reference: every column (select=None must give exactly the final preparation)
all_columns_run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
print("same as the final preparation:",
      bool((all_columns_run[["f1", "accuracy"]] == final_run[["f1", "accuracy"]]).all().all()))

widths = []
for train_idx, val_idx in splits:
    train, test = prepare_final(frame.iloc[train_idx], frame.iloc[val_idx])
    numeric, categorical = model_columns(train)
    encoder = OneHotEncoder(handle_unknown="ignore").fit(as_object(train[categorical]))
    widths.append(len(numeric) + encoder.get_feature_names_out().size)
print("columns the model sees, per split:", widths)
print(f"feature selection reference: F1 {all_columns_run['f1'].mean():.4f} +- {sem(all_columns_run['f1']):.4f}"
      f" | accuracy {all_columns_run['accuracy'].mean():.4f}")

same as the final preparation: True
columns the model sees, per split: [85, 96, 109, 123, 133]
feature selection reference: F1 0.7481 +- 0.0086 | accuracy 0.7158


## 10. Filter methods

`VarianceThreshold(0)` removes constant columns. `SelectKBest` ranks each encoded column by
its link with the target, using |point-biserial r| for numeric columns and Cramér's V for
one-hot columns. The `CorrelationFilter` walks the same ranking but skips a column with |r|
> 0.70 with one already kept, so with the same k the paired difference between the last two
is what the redundancy rule alone is worth.

`VarianceThreshold(0)` removes nothing, because no column is constant. Among the
`SelectKBest` runs only k=10 leaves the noise, and it is worse (F1 −0.0078); k=20 to 80 tie.
`CorrelationFilter(k=10)` is the only filter above the noise in F1 (+0.0071, better on 5 of
5), and the other k values tie. Its gain is weaker than it looks: accuracy does not improve,
it predicts 1 more often than the reference, k was chosen on the same splits that measure
it, and its columns change from split to split. The redundancy rule matters most at k=10
(+0.0149 against `SelectKBest(k=10)`), which spends its 10 slots on counts and spending
columns that repeat each other.

In [39]:
from collections import Counter

from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.feature_selection import SelectKBest, VarianceThreshold

REDUNDANT = 0.70   # above this, a pair of columns is one fact recorded twice (as in the EDA)
# fit_predict puts numeric columns first and the one-hot after
N_NUMERIC = len(model_columns(frame)[0])


def cramers_v(column, outcome):
    """Chi-square on a contingency table, rescaled to [0, 1]."""
    table = pd.crosstab(column, outcome)
    if min(table.shape) < 2:
        return 0.0, float("nan"), 0.0, float(table.to_numpy().min())
    statistic, p, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return statistic, p, v, expected.min()


def encoded_effect_scores(matrix, outcome):
    """One [0, 1] effect per encoded column.

    |point-biserial r| for numeric columns and Cramér's V for one-hot columns, in the order
    of fit_predict: the N_NUMERIC numeric columns first, the one-hot after. MinMaxScaler
    changes neither (it is a linear rescaling) and keeps the one-hot at 0/1.
    """
    matrix = np.asarray(matrix, dtype=float)
    dummy_block = matrix[:, N_NUMERIC:]
    if not np.isin(dummy_block, [0.0, 1.0]).all():
        raise ValueError("Cramer's V requires the 0/1 one-hot dummy block")
    target = np.asarray(outcome, dtype=int)
    scores = []
    for position in range(matrix.shape[1]):
        values = matrix[:, position]
        if position >= N_NUMERIC:
            effect = cramers_v(pd.Series(values), pd.Series(target))[2]
        elif values.min() == values.max():
            effect = 0.0   # constant column in this training split: related to nothing
        else:
            effect = abs(pointbiserialr(target, values).statistic)
        scores.append(effect)
    return np.nan_to_num(np.asarray(scores, dtype=float))


class CorrelationFilter:
    """Rank by relevance, then skip columns redundant with one already kept.

    With `redundancy=None` this is SelectKBest; with `k=None` there is no budget and every
    column that repeats nothing already kept survives. `transform` and
    `get_feature_names_out` are there for fit_predict and evaluate.
    """

    def __init__(self, k, redundancy=None):
        self.k = k
        self.redundancy = redundancy

    def fit(self, frame, outcome):
        self.names_ = np.asarray(frame.columns)
        matrix = frame.to_numpy(dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            relevance = encoded_effect_scores(matrix, outcome)
            between = np.nan_to_num(np.abs(np.corrcoef(matrix, rowvar=False)))
        keep = []
        for candidate in np.argsort(relevance)[::-1]:
            if len(keep) == self.k:
                break
            if self.redundancy is not None and any(
                    between[candidate, chosen] > self.redundancy for chosen in keep):
                continue
            keep.append(int(candidate))
        self.support_ = np.zeros(matrix.shape[1], dtype=bool)
        self.support_[keep] = True
        return self

    def get_support(self):
        return self.support_

    def transform(self, frame):
        return np.asarray(frame)[:, self.support_]

    def get_feature_names_out(self):
        return self.names_[self.support_]


def kept_in_most(run):
    """The columns kept on more than half the splits, with the count."""
    kept = Counter(column for columns in run["kept"] for column in columns)
    return [f"{column} {n}/{len(run)}" for column, n in kept.most_common() if n > len(run) / 2]


def filter_table(runs, metric=METRIC):
    """compare() against all_columns_run, with the mean number of columns and 2 x SEM."""
    table = compare(runs, all_columns_run, metric=metric)
    table.insert(0, "columns", [runs[name]["columns"].mean() for name in table.index])
    table["2 x SEM"] = 2 * table["SEM"]
    return table


print(f"{N_NUMERIC} numeric columns first; redundancy above {REDUNDANT}")

29 numeric columns first; redundancy above 0.7


In [40]:
# VarianceThreshold(0): removes the columns that are constant in each training split
filter_runs = {"VarianceThreshold(0)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                                encoder=ENCODER_FINAL,
                                                select=VarianceThreshold(0.0))}
removed = [width - n for width, n in zip(widths, filter_runs["VarianceThreshold(0)"]["columns"])]
print("columns per split:", widths)
print("columns removed per split:", removed)

columns per split: [85, 96, 109, 123, 133]
columns removed per split: [0, 0, 0, 0, 0]


In [41]:
# relevance: SelectKBest with encoded_effect_scores
K_VALUES = [10, 20, 40, 60, 80]
for k in K_VALUES:
    filter_runs[f"SelectKBest(k={k})"] = evaluate(
        prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
        select=SelectKBest(encoded_effect_scores, k=k))
filter_table({name: filter_runs[name] for name in filter_runs if "KBest" in name}).round(4)

,columns,f1,accuracy,vs base,SEM,better on,2 x SEM
SelectKBest(k=10),10.0,0.740249,0.704638,-0.007827,0.001987,0 of 5,0.003975
SelectKBest(k=20),20.0,0.753311,0.718924,0.005235,0.004309,4 of 5,0.008619
SelectKBest(k=40),40.0,0.746214,0.715399,-0.001862,0.004052,2 of 5,0.008104
SelectKBest(k=60),60.0,0.747887,0.718367,-0.000189,0.004401,3 of 5,0.008802
SelectKBest(k=80),80.0,0.744004,0.713915,-0.004072,0.002225,1 of 5,0.00445


In [42]:
# redundancy: CorrelationFilter skips a column with |r| > 0.70 with one already kept;
# compared with SelectKBest with the same k
for k in K_VALUES + [None]:
    filter_runs[f"CorrelationFilter(k={k})"] = evaluate(
        prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
        select=CorrelationFilter(k, redundancy=REDUNDANT))

skip_rows = {}
for k in K_VALUES:
    pair = compare({f"k={k}": filter_runs[f"CorrelationFilter(k={k})"]},
                   filter_runs[f"SelectKBest(k={k})"])
    pair["2 x SEM"] = 2 * pair["SEM"]
    skip_rows[f"k={k}"] = pair.loc[f"k={k}", ["vs base", "SEM", "2 x SEM", "better on"]]
print("redundancy rule: CorrelationFilter against SelectKBest with the same k (F1)")
print(pd.DataFrame(skip_rows).T.round(4).to_string(), "\n")
print("CorrelationFilter(k=None) columns, per split:",
      list(filter_runs["CorrelationFilter(k=None)"]["columns"]))

redundancy rule: CorrelationFilter against SelectKBest with the same k (F1)
       vs base       SEM   2 x SEM better on
k=10  0.014889  0.001894  0.003788    5 of 5
k=20 -0.003781  0.004377  0.008754    1 of 5
k=40  0.003059  0.003643  0.007286    2 of 5
k=60 -0.003169  0.007495  0.014989    3 of 5
k=80  0.001651  0.004884  0.009768    3 of 5 

CorrelationFilter(k=None) columns, per split: [66, 76, 88, 101, 113]


In [43]:
# every filter against all_columns_run (all columns), on the same 5 splits
for metric in ["f1", "accuracy"]:
    print(f"[{metric}] against all columns (F1 {all_columns_run['f1'].mean():.4f},"
          f" accuracy {all_columns_run['accuracy'].mean():.4f})")
    print(filter_table(filter_runs, metric=metric).round(4).to_string(), "\n")

# the columns kept on most splits (only the narrowest filters, to stay readable)
for name in ["SelectKBest(k=10)", "SelectKBest(k=20)", "CorrelationFilter(k=10)"]:
    usual = kept_in_most(filter_runs[name])
    print(f"{name}: {len(usual)} columns on 3 or more splits")
    print("   ", ", ".join(usual), "\n")

# k=None keeps almost everything, so show what it removes for repeating another column;
# VarianceThreshold(0) removed nothing, so its `kept` is the list of every column
dropped = Counter(column
                  for every, kept in zip(filter_runs["VarianceThreshold(0)"]["kept"],
                                         filter_runs["CorrelationFilter(k=None)"]["kept"])
                  for column in set(every) - set(kept))
print("CorrelationFilter(k=None) removes, for repeating a column already kept:")
print("   ", ", ".join(f"{c} {n}/{len(splits)}" for c, n in dropped.most_common()
                       if n > len(splits) / 2))

[f1] against all columns (F1 0.7481, accuracy 0.7158)
                           columns        f1  accuracy   vs base       SEM better on   2 x SEM
VarianceThreshold(0)         109.2  0.748076   0.71577       0.0       0.0    0 of 5       0.0
SelectKBest(k=10)             10.0  0.740249  0.704638 -0.007827  0.001987    0 of 5  0.003975
SelectKBest(k=20)             20.0  0.753311  0.718924  0.005235  0.004309    4 of 5  0.008619
SelectKBest(k=40)             40.0  0.746214  0.715399 -0.001862  0.004052    2 of 5  0.008104
SelectKBest(k=60)             60.0  0.747887  0.718367 -0.000189  0.004401    3 of 5  0.008802
SelectKBest(k=80)             80.0  0.744004  0.713915 -0.004072  0.002225    1 of 5   0.00445
CorrelationFilter(k=10)       10.0  0.755139    0.7141  0.007062  0.002193    5 of 5  0.004385
CorrelationFilter(k=20)       20.0   0.74953  0.714842  0.001454  0.002807    4 of 5  0.005615
CorrelationFilter(k=40)       40.0  0.749273  0.716141  0.001197  0.001952    3 of 5  0.003

In [44]:
# why CorrelationFilter(k=10) wins in F1: % of purchases predicted as 1
def share_of_ones(select):
    """% of predictions equal to 1 on each split, with the final preparation and `select`."""
    return [fit_predict(prepare_final, frame.iloc[t], frame.iloc[v], SCALER_FINAL,
                        ENCODER_FINAL, None, select).mean() for t, v in splits]


ones = pd.DataFrame({"all columns": share_of_ones(None),
                     "CorrelationFilter(k=10)": share_of_ones(CorrelationFilter(10, REDUNDANT)),
                     "actual repeat rate": [frame[TARGET].iloc[v].mean() for _, v in splits]})
print((100 * ones).round(1).to_string(), "\n\nmean:", (100 * ones.mean()).round(1).to_dict())

   all columns  CorrelationFilter(k=10)  actual repeat rate
0         69.3                     64.4                58.4
1         55.4                     58.1                53.4
2         53.2                     62.0                57.6
3         50.4                     56.7                52.0
4         56.4                     63.0                60.4 

mean: {'all columns': 56.9, 'CorrelationFilter(k=10)': 60.8, 'actual repeat rate': 56.4}


## 11. Targeted checks

Here we test the columns that the EDA and the preparation left in doubt, and the families of copies
the EDA found (23 pairs above 0.70). In each family one column stays, chosen by its meaning;
choosing by its link with the target on the whole training set would look at the validation
rows. We keep `Total a Pagar` for the invoice totals, because the others are parts of it,
the 90-day window for counts and spending, because the target uses the same window, and the
means for the history columns. The rule: a removal that ties goes, because the model
ends up with fewer columns, and a removal that is worse stays. F1 decides, and a removal
that makes accuracy worse beyond the noise also stays.

Every removal ties in F1 and none makes accuracy worse, so all of them go. `random_noise`
measures nothing. `transport_method` takes 60 of the 109 one-hot columns with it and
improves accuracy (+0.0085). `commercial_zone` is barely related to the target (V 0.06). The
four families together improve F1 (+0.0045, better on 5 of 5). Removing all 14 columns takes
the model from 109 to 33 columns and ties on both metrics (F1 +0.0028, accuracy +0.0006).
The kept column of each family was chosen before measuring, so this result is a confirmation
and was not tuned on these splits.

In [45]:
families = {
    "invoice: Total a Pagar only": ["Total Liquido", "Total Bruto", "Total Imp_", "Custo"],
    "counts: purchase_count_90d only": ["purchase_count_30d", "purchase_count_180d",
                                        "purchase_count_365d", "prior_purchase_count"],
    "spending: spend_90d only": ["spend_180d"],
    "history: means only": ["prior_std_purchase_value", "prior_std_gap_days"],
}
checks = {
    "without random_noise": ["random_noise"],
    "without transport_method": ["transport_method"],
    "without commercial_zone": ["commercial_zone"],
    **families,
    "the 4 families together": [c for columns in families.values() for c in columns],
}

# prepare_final, then remove `columns`, before the one-hot and the model
from preprocessing import prepare_without

numeric, categorical = model_columns(frame)
pd.DataFrame({"removes": [", ".join(columns) for columns in checks.values()],
              "columns before the one-hot": [len(columns) for columns in checks.values()],
              "all exist": [set(columns) <= set(numeric + categorical)
                            for columns in checks.values()]},
             index=list(checks))

,removes,columns before the one-hot,all exist
without random_noise,random_noise,1,True
without transport_method,transport_method,1,True
without commercial_zone,commercial_zone,1,True
invoice: Total a Pagar only,"Total Liquido, Total Bruto, Total Imp_, Custo",4,True
counts: purchase_count_90d only,"purchase_count_30d, purchase_count_180d, purchase_count_365d, prior_purchase_count",4,True
spending: spend_90d only,spend_180d,1,True
history: means only,"prior_std_purchase_value, prior_std_gap_days",2,True
the 4 families together,"Total Liquido, Total Bruto, Total Imp_, Custo, purchase_count_30d, purchase_count_180d, purchase_count_365d, prior_purchase_count, spend_180d, prior_std_purchase_value, prior_std_gap_days",11,True


In [46]:
# each check against all_columns_run (all columns); "columns" is the mean the model sees after
# the one-hot (all: 109.2); verdict: better / worse beyond 2 x SEM, otherwise tie
from preprocessing import verdict


def width(prepare):
    """Columns the model sees after the one-hot, averaged over the 5 splits."""
    total = []
    for train_idx, val_idx in splits:
        train, _ = prepare(frame.iloc[train_idx], frame.iloc[val_idx])
        numeric, categorical = model_columns(train)
        total.append(len(numeric) + sum(train[c].nunique() for c in categorical))
    return np.mean(total)


check_runs = {name: evaluate(prepare_without(columns), frame, splits, scaler=SCALER_FINAL,
                             encoder=ENCODER_FINAL)
              for name, columns in checks.items()}
check_tables = {}
for metric in ["f1", "accuracy"]:
    table = compare(check_runs, all_columns_run, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["verdict"] = verdict(table)
    if metric == "f1":
        table.insert(0, "columns", [width(prepare_without(c)) for c in checks.values()])
    check_tables[metric] = table
    print(f"[{metric}] against all columns (F1 {all_columns_run['f1'].mean():.4f},"
          f" accuracy {all_columns_run['accuracy'].mean():.4f})")
    shown = [c for c in ["columns", metric, "vs base", "SEM", "2 x SEM", "better on", "verdict"]
             if c in table]
    print(table[shown].round(4).to_string(), "\n")

[f1] against all columns (F1 0.7481, accuracy 0.7158)
                                 columns        f1   vs base       SEM   2 x SEM better on verdict
without random_noise               108.2  0.749463  0.001387  0.001143  0.002287    3 of 5     tie
without transport_method            48.8  0.752546   0.00447  0.003436  0.006872    4 of 5     tie
without commercial_zone            105.8  0.748792  0.000716  0.000928  0.001855    2 of 5     tie
invoice: Total a Pagar only        105.2  0.747326  -0.00075   0.00116   0.00232    2 of 5     tie
counts: purchase_count_90d only    105.2  0.751496   0.00342  0.001999  0.003998    4 of 5     tie
spending: spend_90d only           108.2   0.74958  0.001504  0.001123  0.002246    3 of 5     tie
history: means only                107.2  0.748114  0.000037  0.001595  0.003191    4 of 5     tie
the 4 families together             98.2  0.752616   0.00454   0.00214  0.004281    5 of 5  better 

[accuracy] against all columns (F1 0.7481, accuracy 0

In [47]:
# one decision per row: a removal that ties goes (fewer columns); one that is
# worse in F1 or in accuracy stays
check_decisions = pd.DataFrame({
    "F1": check_tables["f1"]["verdict"],
    "accuracy": check_tables["accuracy"]["verdict"],
})
check_decisions["decision"] = np.where(
    (check_decisions["F1"] != "worse") & (check_decisions["accuracy"] != "worse"), "remove", "keep")
print(check_decisions.to_string(), "\n")

# everything removed, together: choices that are good alone may not add up; "the 4 families
# together" is already covered by the families, so it is not counted twice
to_drop = sorted({c for name, columns in checks.items()
                  if name != "the 4 families together" and check_decisions.loc[name, "decision"] == "remove"
                  for c in columns})
check_runs["everything removed, together"] = evaluate(prepare_without(to_drop), frame, splits,
                                                      scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
print(f"everything removed ({len(to_drop)} columns before the one-hot): {', '.join(to_drop)}")
print(f"the model now sees {width(prepare_without(to_drop)):.1f} columns on average (all: 109.2)\n")
for metric in ["f1", "accuracy"]:
    joint = compare({"everything removed, together": check_runs["everything removed, together"]},
                    all_columns_run, metric=metric)
    joint["2 x SEM"] = 2 * joint["SEM"]
    joint["verdict"] = verdict(joint)
    print(f"[{metric}]", joint[[metric, "vs base", "SEM", "2 x SEM", "better on", "verdict"]]
         .round(4).to_string(header=metric == "f1"))

                                     F1 accuracy decision
without random_noise                tie      tie   remove
without transport_method            tie   better   remove
without commercial_zone             tie      tie   remove
invoice: Total a Pagar only         tie      tie   remove
counts: purchase_count_90d only     tie      tie   remove
spending: spend_90d only            tie      tie   remove
history: means only                 tie      tie   remove
the 4 families together          better      tie   remove 

everything removed (14 columns before the one-hot): Custo, Total Bruto, Total Imp_, Total Liquido, commercial_zone, prior_purchase_count, prior_std_gap_days, prior_std_purchase_value, purchase_count_180d, purchase_count_30d, purchase_count_365d, random_noise, spend_180d, transport_method
the model now sees 33.4 columns on average (all: 109.2)

[f1]                                     f1   vs base       SEM   2 x SEM better on verdict
everything removed, together  0.750832

In [48]:
from preprocessing import DROPPED_COLUMNS
assert DROPPED_COLUMNS == to_drop   # preprocessing.py holds exactly the columns decided here
print(f"targeted checks: {len(DROPPED_COLUMNS)} columns to remove: {', '.join(DROPPED_COLUMNS)}")

targeted checks: 14 columns to remove: Custo, Total Bruto, Total Imp_, Total Liquido, commercial_zone, prior_purchase_count, prior_std_gap_days, prior_std_purchase_value, purchase_count_180d, purchase_count_30d, purchase_count_365d, random_noise, spend_180d, transport_method


## 12. Wrapper: RFE

RFE judges columns through a trained model: it trains the logistic regression, removes the
column with the smallest absolute weight, trains again, and repeats until k columns remain.
We only use RFE because RFECV and `SequentialFeatureSelector` validate internally, and with
the preparation done outside them that would leak. We do not know k in advance, so we
measure 5, 10, 15 and so on up to the narrowest split (85 columns), and keep the k with the
best mean F1, the smallest one on an exact tie.

RFE(k=25) reaches F1 0.7546 (+0.0065, 2 × SEM = 0.0028, better on 5 of 5), with accuracy in
the noise. The curve is flat, though: every k from 15 to 45 scores between 0.750 and 0.755,
and k was chosen on the same splits that measure it. It also predicts 1 more often (59%
against 57%), and 12 of its 23 most frequent columns are levels of `transport_method`, plus
a salesperson level with only 3 purchases. RFE keeps these rare levels because their extreme
repeat rates give them large weights, so the model may be memorising them.

In [49]:
from sklearn.feature_selection import RFE

rfe_widths = list(range(5, min(widths) + 1, 5))
rfe_runs = {k: evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
                        select=RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                   n_features_to_select=k))
            for k in rfe_widths}

rfe_curve = pd.DataFrame({
    "k": rfe_widths,
    "mean F1": [rfe_runs[k]["f1"].mean() for k in rfe_widths],
    "F1 SEM": [sem(rfe_runs[k]["f1"]) for k in rfe_widths],
    "mean accuracy": [rfe_runs[k]["accuracy"].mean() for k in rfe_widths],
})
best_f1 = rfe_curve["mean F1"].max()
chosen_k = int(rfe_curve.loc[rfe_curve["mean F1"] == best_f1, "k"].min())
rfe_curve["chosen"] = rfe_curve["k"] == chosen_k
print(rfe_curve.round(4).to_string(index=False))
print(f"\nchosen k = {chosen_k}, by mean F1; on an exact tie the smallest")
print(f"reference (all columns, 85 to 133): F1 {all_columns_run['f1'].mean():.4f}")

 k  mean F1  F1 SEM  mean accuracy  chosen
 5   0.7455  0.0092         0.7078   False
10   0.7490  0.0099         0.7106   False
15   0.7534  0.0092         0.7139   False
20   0.7532  0.0080         0.7152   False
25   0.7546  0.0084         0.7174    True
30   0.7530  0.0088         0.7167   False
35   0.7505  0.0085         0.7167   False
40   0.7503  0.0082         0.7191   False
45   0.7516  0.0092         0.7226   False
50   0.7477  0.0094         0.7173   False
55   0.7465  0.0110         0.7180   False
60   0.7441  0.0108         0.7150   False
65   0.7427  0.0105         0.7147   False
70   0.7431  0.0112         0.7141   False
75   0.7438  0.0110         0.7141   False
80   0.7419  0.0115         0.7122   False
85   0.7424  0.0095         0.7115   False

chosen k = 25, by mean F1; on an exact tie the smallest
reference (all columns, 85 to 133): F1 0.7481


In [50]:
# the chosen k against all columns, and the same checks as the filters
rfe_best = {f"RFE(k={chosen_k})": rfe_runs[chosen_k]}
for metric in ["f1", "accuracy"]:
    table = compare(rfe_best, all_columns_run, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["verdict"] = verdict(table)
    print(f"[{metric}]", table[[metric, "vs base", "SEM", "2 x SEM", "better on", "verdict"]]
         .round(4).to_string(header=metric == "f1"))

ones_rfe = pd.DataFrame({
    "all columns": share_of_ones(None),
    f"RFE(k={chosen_k})": share_of_ones(RFE(LogisticRegression(max_iter=1000,
                                                                random_state=RANDOM_STATE),
                                            n_features_to_select=chosen_k)),
    "actual repeat rate": [frame[TARGET].iloc[v].mean() for _, v in splits],
})
print("\n% of purchases predicted as 1, per split:")
print((100 * ones_rfe).round(1).to_string())

usual = kept_in_most(rfe_runs[chosen_k])
print(f"\nRFE(k={chosen_k}): {len(usual)} columns on 3 or more splits")
print("   ", ", ".join(usual))

[f1]                  f1   vs base       SEM   2 x SEM better on verdict
RFE(k=25)  0.754581  0.006505  0.001414  0.002829    5 of 5  better
[accuracy] RFE(k=25)  0.71744  0.00167  0.004375  0.008749  3 of 5  tie

% of purchases predicted as 1, per split:
   all columns  RFE(k=25)  actual repeat rate
0         69.3       66.9                58.4
1         55.4       60.1                53.4
2         53.2       55.8                57.6
3         50.4       53.0                52.0
4         56.4       60.1                60.4

RFE(k=25): 23 columns on 3 or more splits
    Total Imp_ 5/5, Total Desc_ Linha 5/5, prior_mean_purchase_value 5/5, prior_max_purchase_value 5/5, prior_std_gap_days 5/5, purchase_count_90d 5/5, purchase_count_365d 5/5, salesperson_VLOOKUP Master 8 5/5, transport_method_Maybe Tomorrow Delivery 11 5/5, transport_method_Maybe Tomorrow Delivery 18 5/5, transport_method_Maybe Tomorrow Delivery 68 5/5, transport_method_Maybe Tomorrow Delivery 39 4/5, transport_method_M

In [51]:
RFE_K = chosen_k
print(f"RFE with k={RFE_K} goes to the final comparison (F1 {rfe_runs[RFE_K]['f1'].mean():.4f})")

RFE with k=25 goes to the final comparison (F1 0.7546)


## 13. Embedded methods

Embedded methods select columns inside the model itself. `SelectFromModel` trains the model
and keeps columns by its `threshold`. The L1-penalised logistic regression sets the weights
of weak columns to zero, and `threshold=1e-8` keeps the ones still above zero (C=0.1). The
decision tree ranks columns by importance, and `threshold="median"` keeps the more important
half.

Both tie with all columns. L1 keeps only 17.6 columns on average, a sixth of them, and loses
nothing, so it joins the vote. The tree keeps `random_noise` on all 5 splits: a tree without
a depth limit splits on any column, even pure noise, and here its importances do not
separate useful columns from useless ones.

In [52]:
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier


def l1_selector():
    """The L1 selector, new on every call."""
    return SelectFromModel(LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1,
                                              max_iter=1000, random_state=RANDOM_STATE),
                           threshold=1e-8)


def tree_selector():
    """The tree-importance selector, new on every call."""
    return SelectFromModel(DecisionTreeClassifier(random_state=RANDOM_STATE), threshold="median")


embedded_runs = {"L1 (C=0.1)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                        encoder=ENCODER_FINAL, select=l1_selector()),
                 "tree (median)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                           encoder=ENCODER_FINAL, select=tree_selector())}
for metric in ["f1", "accuracy"]:
    table = compare(embedded_runs, all_columns_run, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["verdict"] = verdict(table)
    if metric == "f1":
        table.insert(0, "columns", [embedded_runs[n]["columns"].mean() for n in table.index])
    shown = [c for c in ["columns", metric, "vs base", "SEM", "2 x SEM", "better on", "verdict"]
             if c in table]
    print(f"[{metric}] against all columns (F1 {all_columns_run['f1'].mean():.4f})")
    print(table[shown].round(4).to_string(), "\n")

[f1] against all columns (F1 0.7481)
               columns        f1   vs base       SEM   2 x SEM better on verdict
L1 (C=0.1)        17.6  0.749786   0.00171  0.003742  0.007483    3 of 5     tie
tree (median)     55.0  0.750247  0.002171  0.003453  0.006906    3 of 5     tie 

[accuracy] against all columns (F1 0.7481)
               accuracy   vs base       SEM   2 x SEM better on verdict
L1 (C=0.1)     0.714842 -0.000928  0.005756  0.011512    3 of 5     tie
tree (median)  0.718738  0.002968   0.00253  0.005061    3 of 5     tie 



In [53]:
# which columns each method keeps, and how often it predicts 1
for name in embedded_runs:
    usual = kept_in_most(embedded_runs[name])
    print(f"{name}: {len(usual)} columns on 3 or more splits")
    print("   ", ", ".join(usual), "\n")

ones_embedded = pd.DataFrame({
    "all columns": share_of_ones(None),
    "L1 (C=0.1)": share_of_ones(l1_selector()),
    "tree (median)": share_of_ones(tree_selector()),
    "actual repeat rate": [frame[TARGET].iloc[v].mean() for _, v in splits],
})
print("% of purchases predicted as 1, mean of the 5 splits:")
print((100 * ones_embedded.mean()).round(1).to_string())

L1 (C=0.1): 20 columns on 3 or more splits
    Total Desc_ Linha 5/5, purchase_count_90d 5/5, spend_90d 5/5, purchase_count_365d 5/5, transport_method_Maybe Tomorrow Delivery 72 5/5, warehouse_Final Excel Warehouse 1 5/5, prior_max_purchase_value 4/5, prior_mean_gap_days 4/5, prior_std_gap_days 4/5, salesperson_VLOOKUP Master 4 4/5, transport_method_Maybe Tomorrow Delivery 68 4/5, document_series_Ctrl Z Series 3 3/5, transport_method_Maybe Tomorrow Delivery 53 3/5, Total Imp_ 3/5, N_ Detalhes 3/5, spend_30d 3/5, payment_type_Maybe Tomorrow Payment 7 3/5, payment_type_Maybe Tomorrow Payment 9 3/5, transport_method_Maybe Tomorrow Delivery 59 3/5, transport_method_Maybe Tomorrow Delivery 73 3/5 

tree (median): 56 columns on 3 or more splits
    Total a Pagar 5/5, Total Bruto 5/5, Total Liquido 5/5, Total Imp_ 5/5, Total Portes 5/5, Total Desc_ Global 5/5, Total Desc_ Linha 5/5, Custo 5/5, N_ Detalhes 5/5, invoice_count_current_purchase 5/5, days_since_previous_purchase 5/5, customer_tenu

## 14. Combining methods and the final feature set

For the vote, one method from each family is fitted on the same training rows, and a column
stays when at least 2 of the 3 keep it. The filter is `CorrelationFilter(k=40)` (the widest
one that ties), the wrapper is RFE(k=25) and the embedded method is L1. We then compare
every candidate side by side, including RFE(k=25) after removing the 14 columns of the
targeted checks: if its gain came from the rare levels of `transport_method`, it should
disappear without them.

The final feature set removes the 14 columns of the targeted checks before the one-hot. It
ties with all columns on both metrics (F1 +0.0028, accuracy +0.0006) with a third of the
columns (33 instead of 109). It removes the columns we already knew were a problem:
`random_noise`, `transport_method` (80 levels, 51 of them with fewer than 30 purchases),
`commercial_zone` (21% of the test set is in zones the training data never saw) and the
copies in the redundant families. Because those columns were chosen by meaning before
measuring, the result is not optimistic. RFE(k=25) and `CorrelationFilter(k=10)` score
higher in F1, but their k was chosen on the same splits that measure them, part of their
gain comes from predicting 1 more often, and the RFE gain disappears without
`transport_method` (+0.0018, noise).

`preprocessing.py` saves this as `prepare_selected = prepare_without(DROPPED_COLUMNS)` with
`SELECT_FINAL = None`, and the modelling notebook uses it.

In [54]:
class Vote:
    """A column stays when at least `minimum` of the `members` keep it.

    `transform` and `get_feature_names_out` are added for fit_predict and evaluate (as in
    CorrelationFilter).
    """

    def __init__(self, members, minimum):
        self.members = members
        self.minimum = minimum

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        self.decisions_ = pd.DataFrame(
            {name: member.fit(frame, outcome).get_support()
             for name, member in self.members.items()},
            index=frame.columns,
        )
        self.support_ = (self.decisions_.sum(axis=1) >= self.minimum).to_numpy()
        self.decisions_["kept"] = self.support_
        return self

    def get_support(self):
        return self.support_

    def transform(self, frame):
        return np.asarray(frame)[:, self.support_]

    def get_feature_names_out(self):
        return self.feature_names_in_[self.support_]


def vote_selector():
    """The three methods of the vote, new on every call."""
    return Vote({"filter": CorrelationFilter(40, redundancy=REDUNDANT),
                 "wrapper": RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                n_features_to_select=RFE_K),
                 "embedded": l1_selector()},
                minimum=2)


vote_run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
                    select=vote_selector())
print(f"vote: {vote_run['columns'].mean():.1f} columns on average,"
      f" F1 {vote_run['f1'].mean():.4f}, accuracy {vote_run['accuracy'].mean():.4f}")
usual = kept_in_most(vote_run)
print(f"{len(usual)} columns on 3 or more splits:", ", ".join(usual))

vote: 24.2 columns on average, F1 0.7484, accuracy 0.7158
25 columns on 3 or more splits: Total Desc_ Linha 5/5, N_ Detalhes 5/5, prior_max_purchase_value 5/5, prior_std_gap_days 5/5, purchase_count_90d 5/5, purchase_count_365d 5/5, transport_method_Maybe Tomorrow Delivery 11 5/5, transport_method_Maybe Tomorrow Delivery 68 5/5, transport_method_Maybe Tomorrow Delivery 72 5/5, transport_method_Maybe Tomorrow Delivery 39 4/5, transport_method_Maybe Tomorrow Delivery 80 4/5, transport_method_Maybe Tomorrow Delivery 9 4/5, prior_mean_gap_days 4/5, warehouse_Final Excel Warehouse 1 4/5, spend_90d 3/5, salesperson_VLOOKUP Master 8 3/5, transport_method_Maybe Tomorrow Delivery 50 3/5, transport_method_Maybe Tomorrow Delivery 53 3/5, Total Imp_ 3/5, payment_type_Maybe Tomorrow Payment 7 3/5, payment_type_Maybe Tomorrow Payment 9 3/5, transport_method_Maybe Tomorrow Delivery 22 3/5, transport_method_Maybe Tomorrow Delivery 48 3/5, transport_method_Maybe Tomorrow Delivery 59 3/5, transport_meth

In [55]:
# every candidate side by side, against all columns, on the same 5 splits
rfe_without_dropped = evaluate(prepare_without(DROPPED_COLUMNS), frame, splits, scaler=SCALER_FINAL,
                        encoder=ENCODER_FINAL,
                        select=RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                   n_features_to_select=RFE_K))
candidates = {
    "CorrelationFilter(k=10)": filter_runs["CorrelationFilter(k=10)"],
    "RFE(k=25)": rfe_runs[RFE_K],
    "L1": embedded_runs["L1 (C=0.1)"],
    "without the 14 columns": check_runs["everything removed, together"],
    "RFE(k=25) without the 14": rfe_without_dropped,
    "vote 2 of 3": vote_run,
}
widths_of = {name: (run["columns"].mean() if "columns" in run else width(prepare_without(DROPPED_COLUMNS)))
             for name, run in candidates.items()}


def ones_share(prepare, select=None):
    """Mean % of predictions equal to 1 over the 5 splits, for any preparation and selector."""
    return 100 * np.mean([fit_predict(prepare, frame.iloc[t], frame.iloc[v], SCALER_FINAL,
                                      ENCODER_FINAL, None, select).mean() for t, v in splits])


ones_of = {
    "CorrelationFilter(k=10)": ones_share(prepare_final, CorrelationFilter(10, REDUNDANT)),
    "RFE(k=25)": ones_share(prepare_final, RFE(LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE), n_features_to_select=RFE_K)),
    "L1": ones_share(prepare_final, l1_selector()),
    "without the 14 columns": ones_share(prepare_without(DROPPED_COLUMNS)),
    "RFE(k=25) without the 14": ones_share(prepare_without(DROPPED_COLUMNS), RFE(LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE), n_features_to_select=RFE_K)),
    "vote 2 of 3": ones_share(prepare_final, vote_selector()),
}

f1_table = compare(candidates, all_columns_run, metric="f1")
acc_table = compare(candidates, all_columns_run, metric="accuracy")
summary = pd.DataFrame({
    "columns": pd.Series(widths_of),
    "F1": f1_table["f1"],
    "F1 vs all": f1_table["vs base"],
    "F1 2 x SEM": 2 * f1_table["SEM"],
    "F1 better on": f1_table["better on"],
    "F1 verdict": verdict(f1_table),
    "accuracy vs all": acc_table["vs base"],
    "accuracy verdict": verdict(acc_table),
    "% predicted 1": pd.Series(ones_of),
})
print(f"all columns: F1 {all_columns_run['f1'].mean():.4f}, accuracy"
      f" {all_columns_run['accuracy'].mean():.4f}, columns {width(prepare_final):.1f},"
      f" % predicted 1 {100 * ones_embedded['all columns'].mean():.1f}"
      f" (actual rate {100 * ones_embedded['actual repeat rate'].mean():.1f})\n")
print(summary.round(4).to_string())

all columns: F1 0.7481, accuracy 0.7158, columns 109.2, % predicted 1 56.9 (actual rate 56.4)

                          columns        F1 F1 vs all F1 2 x SEM F1 better on F1 verdict accuracy vs all accuracy verdict  % predicted 1
CorrelationFilter(k=10)      10.0  0.755139  0.007062   0.004385       5 of 5     better        -0.00167              tie        60.8163
RFE(k=25)                    25.0  0.754581  0.006505   0.002829       5 of 5     better         0.00167              tie        59.1837
L1                           17.6  0.749786   0.00171   0.007483       3 of 5        tie       -0.000928              tie        57.9963
without the 14 columns       33.4  0.750832  0.002756   0.004701       3 of 5        tie        0.000557              tie        57.8850
RFE(k=25) without the 14     25.0  0.749875  0.001799   0.007448       2 of 5        tie             0.0              tie        57.6438
vote 2 of 3                  24.2  0.748423  0.000346    0.00611       1 of 5      

In [56]:
from preprocessing import prepare_selected
selected_run = check_runs["everything removed, together"]

selection_decisions = pd.DataFrame([
    ("9. Identifiers", "identifiers (ID, customer_id, purchase_date)", "-", "left out: they do not describe the purchase"),
    ("10. Filter methods", "VarianceThreshold, SelectKBest, CorrelationFilter", "CorrelationFilter(k=10) +0.0071",
     "optimistic gain that comes from predicting more 1s; not kept"),
    ("12. RFE", "RFE with k from 5 to 85", "RFE(k=25) +0.0065",
     "depends on rare levels of transport_method; without them, noise; not kept"),
    ("13. Embedded methods", "L1 and tree importance", "tie", "the tree keeps random_noise; not kept"),
    ("11. Targeted checks", "remove noise, problem columns and copies", "ties with 33 columns",
     "kept: the same result, three times fewer columns, chosen by meaning"),
    ("14. Vote", "vote 2 of 3 (filter, RFE, L1)", "tie", "adds nothing to the targeted checks; not kept"),
], columns=["section", "method", "F1 result", "decision"])
print(selection_decisions.to_string(index=False))
print(f"\nfinal feature set: {width(prepare_selected):.1f} columns, F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}"
      f" (all columns: F1 {all_columns_run['f1'].mean():.4f})")
print("removed:", ", ".join(DROPPED_COLUMNS))

             section                                            method                       F1 result                                                                  decision
      9. Identifiers      identifiers (ID, customer_id, purchase_date)                               -                               left out: they do not describe the purchase
  10. Filter methods VarianceThreshold, SelectKBest, CorrelationFilter CorrelationFilter(k=10) +0.0071              optimistic gain that comes from predicting more 1s; not kept
             12. RFE                           RFE with k from 5 to 85               RFE(k=25) +0.0065 depends on rare levels of transport_method; without them, noise; not kept
13. Embedded methods                            L1 and tree importance                             tie                                     the tree keeps random_noise; not kept
 11. Targeted checks          remove noise, problem columns and copies            ties with 33 columns       kept: 

In [57]:
# the final feature set on test.csv: the same checks as the final preparation
train_selected, test_selected = prepare_selected(frame, test_csv)
numeric, categorical = model_columns(train_selected)
pd.Series({
    "columns before the one-hot: numeric / categorical": f"{len(numeric)} / {len(categorical)}",
    "same columns in train and test": list(model_columns(test_selected)[0]) == numeric
                                      and list(model_columns(test_selected)[1]) == categorical,
    "nulls in train / test": f"{train_selected[numeric + categorical].isna().sum().sum()}"
                             f" / {test_selected[numeric + categorical].isna().sum().sum()}",
    "test rows with an unseen level, per column":
        {c: int((~test_selected[c].isin(train_selected[c])).sum()) for c in categorical},
}).to_frame("check")

,check
columns before the one-hot: numeric / categorical,17 / 4
same columns in train and test,True
nulls in train / test,0 / 0
"test rows with an unseen level, per column","{'payment_type': 3, 'salesperson': 0, 'warehouse': 7, 'document_series': 0}"


In [58]:
# the reference for the modelling notebook
print(f"modelling reference: F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}")

modelling reference: F1 0.7508 +- 0.0074, accuracy 0.7163


## Summary

- Impossible values become null and the remaining gaps get the training median or mode. The
  alternatives tie, so the simplest plan stays.
- `log1p` on the 27 long-tailed columns with `MinMaxScaler` is the only change clearly above
  the noise: F1 0.7481 against 0.7321 for the baseline.
- One-hot encoding stays, and neither the new features nor the columns of `clientes.csv`
  help.
- Feature selection removes 14 columns chosen by meaning (noise, problem columns and
  copies). The model goes from 109 to 33 columns with the same F1 (0.7508) and accuracy
  (0.7163). The methods that score higher in F1 do it partly by predicting 1 more often,
  with a k chosen on the same splits that measure them.